In [1]:
# Part 1- Storing Data in SQLite Database
import sqlite3

connection = sqlite3.connect('company.db')
cursor = connection.cursor()

table_info = '''
CREATE TABLE employees(id INTEGER PRIMARY KEY,
name VARCHAR(15),
department VARCHAR(15),
salary INT
)
'''
table_info2 = '''
CREATE TABLE sales(sale_id Integer Primary key,
employee_id INT,
amount INT,
sale_date VARCHAR(15))
'''
cursor.execute(table_info)
cursor.execute(table_info2)


In [2]:
# Insert
insert1 = '''
INSERT INTO Employees VALUES
(1, 'Shreyas', 'AWS', 30000),
(2, 'Rohan', 'IT', 10000),
(3, 'Ram', 'AIDS', 40000),
(4, 'Amit', 'HR', 25000),
(5, 'Priya', 'IT', 35000),
(6, 'Sneha', 'AWS', 45000),
(7, 'Rahul', 'Sales', 28000),
(8, 'Neha', 'AIDS', 32000),
(9, 'Karan', 'HR', 22000),
(10, 'Pooja', 'Sales', 38000)
'''

insert2= '''
INSERT INTO Sales VALUES
(101, 1, 5000, '2026-01-10'),
(102, 2, 3000, '2026-01-12'),
(103, 3, 7000, '2026-01-15'),
(104, 4, 2500, '2026-01-18'),
(105, 5, 6000, '2026-01-20'),
(106, 6, 8000, '2026-01-22'),
(107, 7, 4500, '2026-01-25'),
(108, 8, 5500, '2026-01-28'),
(109, 9, 2000, '2026-02-01'),
(110, 10, 9000, '2026-02-05')
'''

cursor.execute(insert1)
cursor.execute(insert2)



In [3]:
cursor.execute("SELECT * FROM Employees")
print(cursor.fetchall())

cursor.execute("SELECT * FROM Sales")
print(cursor.fetchall())

[(1, 'Shreyas', 'AWS', 30000), (2, 'Rohan', 'IT', 10000), (3, 'Ram', 'AIDS', 40000), (4, 'Amit', 'HR', 25000), (5, 'Priya', 'IT', 35000), (6, 'Sneha', 'AWS', 45000), (7, 'Rahul', 'Sales', 28000), (8, 'Neha', 'AIDS', 32000), (9, 'Karan', 'HR', 22000), (10, 'Pooja', 'Sales', 38000)]
[(101, 1, 5000, '2026-01-10'), (102, 2, 3000, '2026-01-12'), (103, 3, 7000, '2026-01-15'), (104, 4, 2500, '2026-01-18'), (105, 5, 6000, '2026-01-20'), (106, 6, 8000, '2026-01-22'), (107, 7, 4500, '2026-01-25'), (108, 8, 5500, '2026-01-28'), (109, 9, 2000, '2026-02-01'), (110, 10, 9000, '2026-02-05')]


In [4]:
connection.commit()
connection.close()

In [5]:

from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase

# Create SQLAlchemy engine
engine = create_engine("sqlite:///company.db")

# Connect LangChain SQLDatabase
db = SQLDatabase(engine)

# Test connection
print("Available Tables:", db.get_usable_table_names())

# Inspect schema
print(db.get_table_info())


C:\Users\shrey\AppData\Local\Temp\ipykernel_8276\1643198895.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Available Tables: ['employees', 'sales']

CREATE TABLE employees (
	id INTEGER, 
	name VARCHAR(15), 
	department VARCHAR(15), 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Shreyas	AWS	30000
2	Rohan	IT	10000
3	Ram	AIDS	40000
*/


CREATE TABLE sales (
	sale_id INTEGER, 
	employee_id INTEGER, 
	amount INTEGER, 
	sale_date VARCHAR(15), 
	PRIMARY KEY (sale_id)
)

/*
3 rows from sales table:
sale_id	employee_id	amount	sale_date
101	1	5000	2026-01-10
102	2	3000	2026-01-12
103	3	7000	2026-01-15
*/


In [6]:
from sqlalchemy import create_engine, URL


# Create MySQL connection URL
connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="root",
    host="localhost",
    port=3306,
    database="college_db"
)

# Create SQLAlchemy engine
engine = create_engine(connection_url)

# Connect LangChain SQLDatabase
db = SQLDatabase(engine)

# Test connection
print("Available Tables:")
print(db.get_usable_table_names())

# Display schema
print("\nSchema Information:")
print(db.get_table_info())


Available Tables:
['students']

Schema Information:

CREATE TABLE students (
	student_id INTEGER NOT NULL AUTO_INCREMENT, 
	name VARCHAR(50), 
	age INTEGER, 
	department VARCHAR(50), 
	city VARCHAR(50), 
	cgpa DECIMAL(3, 2), 
	PRIMARY KEY (student_id)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from students table:
student_id	name	age	department	city	cgpa
1	Rahul Sharma	21	Computer Science	Nashik	8.20
2	Priya Patil	22	Artificial Intelligence	Pune	8.75
3	Amit Verma	21	Information Technology	Mumbai	7.80
*/


In [12]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
LLM = ChatGroq(model='openai/gpt-oss-20b')
toolkit = SQLDatabaseToolkit(db=db,llm=LLM)
tools = toolkit.get_tools()
for t in tools:
    print(t.name,'-->',t.description)

sql_db_query --> Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.
sql_db_schema --> Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3
sql_db_list_tables --> Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker --> Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


In [24]:
query_tool = next(t for t in tools if t.name == "sql_db_query")
schema_tool = next(t for t in tools if t.name == "sql_db_schema")

print(schema_tool.invoke({"table_names": "students"}))
print('\n')
print(query_tool.invoke({"query": "SELECT COUNT(*) FROM students"}))


CREATE TABLE students (
	student_id INTEGER NOT NULL AUTO_INCREMENT, 
	name VARCHAR(50), 
	age INTEGER, 
	department VARCHAR(50), 
	city VARCHAR(50), 
	cgpa DECIMAL(3, 2), 
	PRIMARY KEY (student_id)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from students table:
student_id	name	age	department	city	cgpa
1	Rahul Sharma	21	Computer Science	Nashik	8.20
2	Priya Patil	22	Artificial Intelligence	Pune	8.75
3	Amit Verma	21	Information Technology	Mumbai	7.80
*/


[(20,)]


In [25]:
#Creating SQL Agent
from langchain_community.agent_toolkits import create_sql_agent
agent = create_sql_agent(llm=LLM,db=db,verbose=True,agent_type='tool-calling')
agent.invoke({
    "input": "Show all students"
})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


students
Invoking: `sql_db_schema` with `{'table_names': 'students'}`



CREATE TABLE students (
	student_id INTEGER NOT NULL AUTO_INCREMENT, 
	name VARCHAR(50), 
	age INTEGER, 
	department VARCHAR(50), 
	city VARCHAR(50), 
	cgpa DECIMAL(3, 2), 
	PRIMARY KEY (student_id)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from students table:
student_id	name	age	department	city	cgpa
1	Rahul Sharma	21	Computer Science	Nashik	8.20
2	Priya Patil	22	Artificial Intelligence	Pune	8.75
3	Amit Verma	21	Information Technology	Mumbai	7.80
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT student_id, name, age, department, city, cgpa FROM students ORDER BY student_id LIMIT 10;'}`


SELECT student_id, name, age, department, city, cgpa FROM students ORDER BY student_id LIMIT 10;
Invoking: `sql_db_query` with `{'query': 'SELECT student_id, name, age, department,

{'input': 'Show all students',
 'output': 'Here are the first 10 students in the database (ordered by `student_id`):\n\n| student_id | name           | age | department          | city   | cgpa |\n|------------|----------------|-----|---------------------|--------|------|\n| 1          | Rahul Sharma   | 21  | Computer Science    | Nashik | 8.20 |\n| 2          | Priya Patil    | 22  | Artificial Intelligence | Pune   | 8.75 |\n| 3          | Amit Verma     | 21  | Information Technology | Mumbai | 7.80 |\n| 4          | Sneha Joshi    | 20  | Data Science        | Nashik | 9.10 |\n| 5          | Rohan Deshmukh | 22  | Computer Science    | Pune   | 7.50 |\n| 6          | Neha Kulkarni  | 21  | Artificial Intelligence | Mumbai | 8.60 |\n| 7          | Karan Shah     | 23  | Information Technology | Surat  | 7.90 |\n| 8          | Pooja More     | 20  | Data Science        | Nashik | 8.95 |\n| 9          | Akash Singh    | 22  | Computer Science    | Delhi  | 7.20 |\n| 10         | Riya

In [27]:
engine = create_engine("sqlite:///company.db")
db = SQLDatabase(engine)
#Creating SQL Agent
from langchain_community.agent_toolkits import create_sql_agent
agent = create_sql_agent(llm=LLM,db=db,verbose=True,agent_type='tool-calling')
res1=agent.invoke({
    "input": "How many employees are there in each department?"
})
res2=agent.invoke({
    "input": "Who has the highest salary?"
})
res3=agent.invoke({
    "input": "What is the total sales amount?"
})
res4=agent.invoke({
    "input": "What is the average salary per department?"
})
print(res1)
print(res2)
print(res3)
print(res4)




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


employees, sales
Invoking: `sql_db_schema` with `{'table_names': 'employees'}`



CREATE TABLE employees (
	id INTEGER, 
	name VARCHAR(15), 
	department VARCHAR(15), 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Shreyas	AWS	30000
2	Rohan	IT	10000
3	Ram	AIDS	40000
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10'}`


SELECT department, COUNT(*) AS employee_count 
FROM employees 
GROUP BY department 
ORDER BY employee_count DESC 
LIMIT 10
Invoking: `sql_db_query` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10'}`


[('Sales', 2), ('IT', 2), ('HR', 2), ('AWS', 2), ('AIDS', 2)]The query returned five rows, each showing a depar

Observations & Insights Write short answers: 
1. Why SQL agents are better than manual SQL generation?
SQL agents gives results based on the prompt and it is easy for non tech person to understand the data
2. Difference between SQL Agent and RAG?
SQL Agent runs the query and display result in natural language while RAG stores data in chunks and retrieves when we asked the relevent query.
3. Risks of allowing unrestricted SQL access?
Yes there might be risk of unrestricted SQL access
